## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            ((data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1]))
            .any("nlat")
            .values
        )

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            ((data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1]))
            .any("nlon")
            .values
        )

    return data.isel(nlon=lon_idx, nlat=lat_idx)


def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x: (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)


def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(["longitude", "latitude"])
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli


def get_eli(rsst, rsst_thresh=0, max_lat=15):
    """compute ELI from tropical SST data"""

    ## get equatorial Pac. SST
    rsst_pac = rsst.sel(longitude=slice(120, 280), latitude=slice(-max_lat, max_lat))

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= rsst_thresh

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    return eli0

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### do loading

In [ ]:
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
SAVE_FNAME = SAVE_FP / "pr.nc"

## load precip
pr = xr.open_dataarray(SAVE_FNAME).compute()
pr = pr.rename({"lon": "longitude", "lat": "latitude"})

## convert m/s to mm/day
mm_per_m = 1e3
s_per_day = 24 * 60 * 60
pr *= mm_per_m * s_per_day

## Analysis

In [ ]:
## get climatology
get_clim = lambda x: x.groupby("time.month").mean(["time", "member"])
clim = get_clim(pr.isel(time=slice(None, 360)))

clim_fma = clim.sel(month=[2, 3, 4]).mean("month")
clim_aso = clim.sel(month=[8, 9, 10]).mean("month")

### Spatial plot

In [ ]:
## specify intervals
dxs = np.array([2])

fig = plt.figure(figsize=(7, 9), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=40, lon_range=(60, 280))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)


cp00 = plot_pr(axs[0, 0], clim_aso, dx=2)
cp00 = plot_pr(axs[1, 0], clim_fma, dx=2)
cp00 = plot_pr_diff(axs[2, 0], clim_aso - clim_fma, dx=2)

## label
axs[0, 0].set_title("(a) FMA", loc="left")
axs[1, 0].set_title("(b) ASO", loc="left")
axs[2, 0].set_title("(c) Difference", loc="left")

## add labels
# add_gridlines(axs)
bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax in axs.flatten():
    ax.set_aspect("auto")
    src.utils.plot_box(ax, lons=[65, 150], lats=[-25, 25], c="k", ls="--")

## colorbars
cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
# fig.colorbar(cp00, ax=axs[0,0], ticks=[-3.6,0,3.6], **cb_kwargs)
# fig.colorbar(cp10, ax=axs[1,0], ticks=[-1.8,0,1.8], **cb_kwargs)
add_gridlines(axs)
plt.show()

In [ ]:
## average over longitudes in australia/Asia
clim_meri = clim.sel(longitude=slice(65, 140)).mean("longitude")

fig, ax = plt.subplots(figsize=(3, 5))
ax.contourf(
    clim_meri.month,
    clim_meri.latitude,
    clim_meri.transpose("latitude", "month"),
    cmap="cmo.rain",
    extend="both",
    levels=np.arange(0, 11, 1),
)

# ax.set_ylim([-25,25])

src.utils.add_vticks([ax], xlines=[3, 9], xticks=[])
ax.set_xticks([3, 9], labels=["Mar", "Sep"])
ax.axhline(0, ls="--", c="gray", lw=0.8)

plt.show()

## Look at variability

### Seasonal cycle

In [ ]:
lon_range = slice(65, 140)
pr_meri = pr.sel(longitude=lon_range).mean("longitude")
pr_s = pr_meri.sel(latitude=slice(-25, 0)).mean("latitude")
pr_n = pr_meri.sel(latitude=slice(0, 25)).mean("latitude")
pr_idx = xr.merge([pr_s.rename("s"), pr_n.rename("n")])

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
for r in ["n", "s"]:
    ax.plot(np.arange(1, 13), get_clim(pr_idx[r]), label=r)

ax.legend()
plt.show()

In [ ]:
pr_idx_ = pr_idx.resample({"time": "QS-FEB"}).mean()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

# ax.scatter(
#     pr_idx_["n"].isel(time=slice(2,2+480,None)),
#     pr_idx_["s"].isel(time=slice(2,2+480,None)),
#     s=2,
# )

ax.scatter(
    pr_idx_["n"].isel(time=slice(2, 2 + 480, 4)),
    pr_idx_["s"].isel(time=slice(4, 4 + 480, 4)),
    # s=2,
)

ax.scatter(
    pr_idx_["n"].isel(time=slice(None, 480)),
    pr_idx_["s"].isel(time=slice(None, 480)),
    s=2,
)

### forced change over time

In [ ]:
## 1. hovmoller of N. hemisphere precip/seasonal cycle (cyc. vs. time)
## 2. "" (lat vs. time for given season)

In [ ]:
pr_meri_seasonal = pr_meri.resample({"time": "QS-FEB"}).mean()

In [ ]:
n_half = 15
n_smooth = int(n_half * 2 + 1)
n_trim = int(n_half * 12)

pr_meri_forced = (
    pr_meri.mean("member")
    .groupby("time.month")
    .map(lambda x: x.rolling({"time": n_smooth}, center=True).mean())
)

## Get forced change
pr_meri_forced = pr_meri_forced.isel(time=slice(n_trim, -n_trim))

## resample to seasonal
pr_meri_seasonal = pr_meri_forced.resample({"time": "QS-FEB"}).mean()

plot aboslute diff

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(5, 8), layout="constrained")

for ax, t_start_idx in zip(
    axs,
    [1, 2, 3, 4],
):

    ## get plot data
    pdata = pr_meri_seasonal.isel(
        time=slice(t_start_idx, None, 4),
    )

    ## update time coord
    pdata = pdata.assign_coords({"time": pdata.time.dt.year.values})
    pdata = pdata.transpose("latitude", ...)

    ## plot data
    ax.contourf(
        pdata.time,
        pdata.latitude,
        pdata,
        levels=np.arange(1, 11, 1),
        cmap="cmo.rain",
        extend="both",
    )

    ax.set_xticks([])
    ax.axhline(0, ls="--", c="k", lw=1)

plt.show()

In [ ]:
delta = lambda x: x - x.isel(time=0)

fig, axs = plt.subplots(4, 1, figsize=(5, 8), layout="constrained")

for ax, t_start_idx in zip(
    axs,
    [1, 2, 3, 4],
):

    ## get plot data
    pdata = pr_meri_seasonal.isel(
        time=slice(t_start_idx, None, 4),
    )

    ## update time coord
    pdata = pdata.assign_coords({"time": pdata.time.dt.year.values})
    pdata = pdata.transpose("latitude", ...)

    ## plot data
    ax.contourf(
        pdata.time,
        pdata.latitude,
        delta(pdata),
        levels=src.utils.make_cb_range(0.5, 0.1),
        cmap="cmo.balance_r",
        extend="both",
    )

    ax.set_xticks([])
    ax.axhline(0, ls="--", c="k", lw=1)

plt.show()

### EOFs

In [ ]:
def unstack_season(x):
    """unstack season"""
    x_ = src.utils.unstack_month_and_year(x)

    ## rename
    x_["season"] = xr.DataArray(
        ["FMA", "MJJ", "ASO", "NDJ"],
        coords=dict(month=x_.month),
    )

    return x_.swap_dims({"month": "season"}).drop_vars("month")


# def prep(x):
#     """prepare data for analysis"""

#     ## resample to seasonal
#     x_ = x.resample({"time": "QS-FEB"}).mean()

#     ## only keep pre-industrial and filter out spinup
#     x_ = x_.sel(time=slice("1049-08", "1851-05"))

#     ## get season as dimension
#     x_ = unstack_season(x_)

#     ## trim to remove NaNs from unstacking
#     x_ = x_.isel(year=slice(1, -1))

#     return x_


def prep(y):
    """prepare data for EOF analysis"""

    ## resample to seasonal
    y_ = y.resample({"time": "QS-FEB"}).mean()

    ## find months
    is_aso = y_.time.dt.month == 8
    is_ndj = y_.time.dt.month == 11
    is_fma = y_.time.dt.month == 2
    in_range = is_aso.values | is_ndj.values | is_fma.values

    ## subset for data
    y_ = y_.isel(time=in_range).isel(time=slice(1, -2))

    ## get year start
    yrs = y_.time.dt.year.values
    mths = y_.time.dt.month.values
    year_start = np.array([y - 1 if (m == 2) else y for (y, m) in zip(yrs, mths)])

    ## get season
    season_dict = {2: "FMA", 8: "ASO", 11: "NDJ"}
    season = np.array([season_dict[m] for m in mths])

    ## create multi-index
    new_idx = pd.MultiIndex.from_arrays([year_start, season], names=["y0", "season"])

    ## convert to xr
    new_idx_xr = xr.Coordinates.from_pandas_multiindex(new_idx, dim="time")

    ## add to original data
    y_ = y_.assign_coords(new_idx_xr)

    return y_.unstack("time")

In [ ]:
## get anoms and resample to seasonal
pr_meri_anom = pr_meri - pr_meri_forced

X_ = prep(pr_meri_anom)
X_clim = prep(pr_meri_forced).mean("y0")

In [ ]:
import xeofs as xe

## specs for EOFs
eofs_kwargs = dict(n_modes=10, standardize=False, use_coslat=True, center=True)

## initialize EOF model
eofs = xe.single.EOF(**eofs_kwargs)

## compute
eofs.fit(X_, dim=["y0", "member"])

normalize so scores have unit standard dev.

In [ ]:
## get number of samples
N = int(len(X_.y0) * len(X_.member))

## do normalization
scores_norm = eofs.scores(normalized=True) * np.sqrt(N)
comps = eofs.components(normalized=True) * np.sqrt(eofs.explained_variance())

## check normalization
print(np.allclose(np.ones(10), (scores_norm).std(["member", "y0"]).values))

## check reconstruction
fn = lambda x: x.sel(latitude=slice(-25, 0)).mean(["latitude"]).sel(season="ASO")
idx = dict(member=0, y0=slice(None, 24))

fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(fn(X_).isel(idx))
ax.plot((fn(comps) * scores_norm.isel(idx)).sum("mode"))
plt.show()

#### Get climatology

To-do: plot sum and difference of AMJ/FMA

In [ ]:
X_clim + comps

In [ ]:
comps

In [ ]:
# specify EOF scale
EOF_SCALE = -2

## get total
total = X_clim + EOF_SCALE * comps.isel(mode=0)

clim_sum = X_clim.mean("season")
eof_sum = comps.sum("season")

fig, axs = plt.subplots(1, 4, figsize=(10, 4), layout="constrained")

for ax, s in zip(axs, ["ASO", "NDJ", "FMA"]):

    ax.set_title(s)

    ## plot clim
    ax.plot(X_clim.sel(season=s), comps.latitude, c="k", label="clim")

    ## plot clim + eof
    # sum_ = X_clim.sel(season=s) + EOF_SCALE * comps.sel(season=s).isel(mode=0)
    ax.plot(total.sel(season=s), comps.latitude, label="clim + EOF")

    # plot EOF
    # ax.plot(4 * comps.isel(mode=0).sel(season=s), comps.latitude)

## plot sum over seasons
axs[-1].plot(X_clim.mean("season"), comps.latitude, c="k", label="clim")
axs[-1].plot(total.mean("season"), comps.latitude)

for ax in axs:
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="gray", lw=0.8)


axs[-1].set_title("Sum")
axs[1].legend()
axs[0].set_yticks([-40, -20, 0, 20, 40])
src.utils.add_vticks(axs, xlines=[0], xticks=[0, 8])
src.utils.set_lims(axs)

Look at sum/difference

In [ ]:
## funcs to get sum/diff
get_diff = lambda x: x.sel(season="FMA") - x.sel(season="ASO")
# get_sum = lambda x : x.sel(season="FMA") + x.sel(season="NDJ") + x.sel(season="ASO")
# get_sum = lambda x : x.sel(season="NDJ")
get_sum = lambda x: x.sum("season")

## difference/sum in clim
dX_clim = get_diff(X_clim)
sum_clim = get_sum(X_clim)

## difference/sum in eof
dX_eof = EOF_SCALE * get_diff(comps).isel(mode=0)
sum_eof = EOF_SCALE * get_sum(comps).isel(mode=0)

fig, axs = plt.subplots(1, 2, figsize=(5, 4), layout="constrained")

for ax, clim_, eof_ in zip(axs, [dX_clim, sum_clim], [dX_eof, sum_eof]):

    ## plot clim
    ax.plot(clim_, comps.latitude, c="k", label="clim")

    ## plot clim + eof
    # sum_ = X_clim.sel(season=s) - 2 * comps.sel(season=s).isel(mode=2)
    ax.plot(clim_ + eof_, eof_.latitude, label="eof")

    # plot EOF
    # ax.plot(4 * comps.isel(mode=0).sel(season=s), comps.latitude)

    ax.set_yticks([])
    ax.axhline(0, ls="--", c="gray", lw=0.8)
    ax.set_title(s)

axs[1].legend()
axs[0].set_yticks([-40, -20, 0, 20, 40])
axs[0].set_title(r"FMA$-$ASO")
axs[1].set_title(r"FMA$+$ASO")
src.utils.add_vticks(axs, xlines=[0], xticks=[0, 8])
src.utils.set_lims(axs)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(
    eofs.scores().isel(mode=0),
    eofs.scores().isel(mode=1),
    s=1,
)

src.utils.add_axes([ax])


plt.show()